# GUIDE Dataset — Baseline Models + Ensemble

Target: `IncidentGrade` (TruePositive / FalsePositive / BenignPositive), 3-class classification.

Model tiers:
1. Floor baseline — Dummy (most-frequent)
2. Simple baselines — Logistic Regression, Decision Tree
3. Strong baselines — Random Forest
4. Gradient boosting — LightGBM, XGBoost, CatBoost
5. Ensemble — soft voting + validation-weighted voting over the 3 boosting models

All models are evaluated on the same held-out official `GUIDE_Test.csv` using accuracy and macro-F1 (macro-F1 matters most here since classes are imbalanced).

In [ ]:
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

t0 = time.time()
def lap(msg):
    print(f'[{time.time() - t0:7.1f}s] {msg}')

## 1. Load data

In [ ]:
matches = glob.glob('/kaggle/input/**/GUIDE_Train.csv', recursive=True)
if not matches:
    raise FileNotFoundError('GUIDE_Train.csv not found under /kaggle/input')
DATA_DIR = os.path.dirname(matches[0])
lap(f'Using DATA_DIR = {DATA_DIR}')

train = pd.read_csv(f'{DATA_DIR}/GUIDE_Train.csv')
test = pd.read_csv(f'{DATA_DIR}/GUIDE_Test.csv')
lap(f'Loaded train={train.shape}, test={test.shape}')

train = train.drop_duplicates().reset_index(drop=True)
lap(f'After dropping duplicates: train={train.shape}')

TARGET = 'IncidentGrade'
train = train.dropna(subset=[TARGET]).reset_index(drop=True)
test = test.dropna(subset=[TARGET]).reset_index(drop=True)
lap(f'After dropping rows with missing target: train={train.shape}, test={test.shape}')

## 2. Feature engineering

Drop pure identifier/hash columns (they don't generalize and risk overfitting on noise). Keep contextual/behavioral columns, derive time features, and summarize `MitreTechniques`.

In [ ]:
ID_LIKE_COLS = ['Id', 'OrgId', 'IncidentId', 'AlertId', 'DeviceId', 'Sha256', 'IpAddress', 'Url',
                'AccountSid', 'AccountUpn', 'AccountObjectId', 'AccountName', 'DeviceName',
                'NetworkMessageId', 'EmailClusterId', 'RegistryKey', 'RegistryValueName',
                'RegistryValueData', 'ApplicationId', 'ApplicationName', 'OAuthApplicationId',
                'ThreatFamily', 'FileName', 'FolderPath', 'ResourceIdName']

CATEGORICAL_COLS = ['Category', 'EntityType', 'EvidenceRole', 'ActionGrouped', 'ActionGranular',
                     'ResourceType', 'Roles', 'OSFamily', 'OSVersion', 'AntispamDirection',
                     'SuspicionLevel', 'LastVerdict', 'MitrePrimary']

NUMERIC_COLS = ['DetectorId', 'AlertTitle', 'CountryCode', 'State', 'City',
                'Hour', 'DayOfWeek', 'MitreCount']


def engineer(df):
    df = df.drop(columns=[c for c in ID_LIKE_COLS if c in df.columns], errors='ignore')
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df['Hour'] = df['Timestamp'].dt.hour
    df['DayOfWeek'] = df['Timestamp'].dt.dayofweek
    df = df.drop(columns=['Timestamp'])

    mitre = df['MitreTechniques'].fillna('')
    df['MitreCount'] = mitre.apply(lambda s: 0 if s == '' else len(s.split(';')))
    df['MitrePrimary'] = mitre.apply(lambda s: s.split(';')[0] if s != '' else 'None')
    df = df.drop(columns=['MitreTechniques'])
    return df

train = engineer(train)
test = engineer(test)
lap(f'Engineered features: train={train.shape}, test={test.shape}')

FEATURES = CATEGORICAL_COLS + NUMERIC_COLS
print('Feature count:', len(FEATURES))
print(FEATURES)

## 3. Encoding

Label-encode categoricals (fit on train, unseen test values map to a reserved "unknown" code). Encode the target the same way.

In [ ]:
cat_maps = {}
for col in CATEGORICAL_COLS:
    train[col] = train[col].fillna('Missing').astype(str)
    test[col] = test[col].fillna('Missing').astype(str)
    uniques = sorted(train[col].unique())
    mapping = {v: i for i, v in enumerate(uniques)}
    unknown_code = len(mapping)
    cat_maps[col] = (mapping, unknown_code)
    train[col] = train[col].map(mapping).astype('int32')
    test[col] = test[col].map(mapping).fillna(unknown_code).astype('int32')

for col in NUMERIC_COLS:
    train[col] = pd.to_numeric(train[col], errors='coerce').fillna(-1)
    test[col] = pd.to_numeric(test[col], errors='coerce').fillna(-1)

target_classes = sorted(train[TARGET].unique())
target_map = {v: i for i, v in enumerate(target_classes)}
print('Target classes:', target_map)

train[TARGET] = train[TARGET].map(target_map)
test = test[test[TARGET].isin(target_map)].reset_index(drop=True)
test[TARGET] = test[TARGET].map(target_map)

X_full, y_full = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
lap(f'Encoded. X_full={X_full.shape}, X_test={X_test.shape}')

## 4. Train / validation split

Stratified split from the training set, used for early stopping and ensemble weighting. Final scoring for every model happens on the untouched official test set.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.1, random_state=RANDOM_STATE, stratify=y_full
)
lap(f'Train/val split: X_train={X_train.shape}, X_val={X_val.shape}')

SAMPLE_SIZE = 1_500_000
if len(X_train) > SAMPLE_SIZE:
    sample_idx, _ = train_test_split(
        X_train.index, train_size=SAMPLE_SIZE, random_state=RANDOM_STATE, stratify=y_train
    )
    X_train_small = X_train.loc[sample_idx]
    y_train_small = y_train.loc[sample_idx]
else:
    X_train_small, y_train_small = X_train, y_train
lap(f'Baseline-tier subsample: {X_train_small.shape}')

## 5. Train & evaluate every model

Each model is wrapped in try/except so one failure doesn't kill the whole run.

In [ ]:
results = []
val_probs = {}
test_probs = {}
n_classes = len(target_map)

def score(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')
    results.append({'model': name, 'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1})
    lap(f'{name}: accuracy={acc:.4f}  macro_f1={macro_f1:.4f}  weighted_f1={weighted_f1:.4f}')

def run_model(name, fit_fn):
    try:
        lap(f'--- {name}: training ---')
        fit_fn()
        lap(f'--- {name}: done ---')
    except Exception as e:
        lap(f'--- {name}: FAILED ({e}) ---')

In [ ]:
def fit_dummy():
    clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
    clf.fit(X_train_small, y_train_small)
    score('Dummy (most_frequent)', y_test, clf.predict(X_test))

run_model('Dummy', fit_dummy)

In [ ]:
def fit_logreg():
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_train_small)
    X_te_scaled = scaler.transform(X_test)
    clf = LogisticRegression(max_iter=200, n_jobs=-1, multi_class='multinomial')
    clf.fit(X_tr_scaled, y_train_small)
    score('Logistic Regression', y_test, clf.predict(X_te_scaled))

run_model('Logistic Regression', fit_logreg)

In [ ]:
def fit_dtree():
    clf = DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE)
    clf.fit(X_train_small, y_train_small)
    score('Decision Tree', y_test, clf.predict(X_test))

run_model('Decision Tree', fit_dtree)

In [ ]:
def fit_rf():
    clf = RandomForestClassifier(
        n_estimators=150, max_depth=16, n_jobs=-1, random_state=RANDOM_STATE
    )
    clf.fit(X_train_small, y_train_small)
    score('Random Forest', y_test, clf.predict(X_test))

run_model('Random Forest', fit_rf)

In [ ]:
def fit_lgbm():
    clf = lgb.LGBMClassifier(
        objective='multiclass', num_class=n_classes, n_estimators=400,
        num_leaves=63, learning_rate=0.05, n_jobs=-1, random_state=RANDOM_STATE
    )
    clf.fit(
        X_train, y_train, eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    val_probs['LightGBM'] = clf.predict_proba(X_val)
    test_probs['LightGBM'] = clf.predict_proba(X_test)
    score('LightGBM', y_test, test_probs['LightGBM'].argmax(axis=1))

run_model('LightGBM', fit_lgbm)

In [ ]:
def fit_xgb():
    clf = xgb.XGBClassifier(
        objective='multi:softprob', num_class=n_classes, n_estimators=400,
        max_depth=8, learning_rate=0.05, tree_method='hist',
        n_jobs=-1, random_state=RANDOM_STATE, early_stopping_rounds=30,
        eval_metric='mlogloss'
    )
    clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_probs['XGBoost'] = clf.predict_proba(X_val)
    test_probs['XGBoost'] = clf.predict_proba(X_test)
    score('XGBoost', y_test, test_probs['XGBoost'].argmax(axis=1))

run_model('XGBoost', fit_xgb)

In [ ]:
def fit_catboost():
    clf = CatBoostClassifier(
        loss_function='MultiClass', iterations=500, depth=8, learning_rate=0.05,
        cat_features=[FEATURES.index(c) for c in CATEGORICAL_COLS],
        random_state=RANDOM_STATE, verbose=False, early_stopping_rounds=30
    )
    clf.fit(X_train, y_train, eval_set=(X_val, y_val))
    val_probs['CatBoost'] = clf.predict_proba(X_val)
    test_probs['CatBoost'] = clf.predict_proba(X_test)
    score('CatBoost', y_test, test_probs['CatBoost'].argmax(axis=1))

run_model('CatBoost', fit_catboost)

## 6. Ensemble

Combine the boosting models that produced probability outputs: simple average (soft voting) and a validation-macro-F1-weighted average.

In [ ]:
def fit_ensembles():
    available = [m for m in ['LightGBM', 'XGBoost', 'CatBoost'] if m in test_probs]
    if len(available) < 2:
        lap('Not enough base models succeeded to build an ensemble.')
        return

    soft_avg = np.mean([test_probs[m] for m in available], axis=0)
    score('Ensemble (soft voting)', y_test, soft_avg.argmax(axis=1))

    val_f1 = {
        m: f1_score(y_val, val_probs[m].argmax(axis=1), average='macro')
        for m in available
    }
    print('Validation macro-F1 per model (ensemble weights):', val_f1)
    weights = np.array([val_f1[m] for m in available])
    weights = weights / weights.sum()
    weighted_avg = np.average([test_probs[m] for m in available], axis=0, weights=weights)
    score('Ensemble (weighted voting)', y_test, weighted_avg.argmax(axis=1))

run_model('Ensemble', fit_ensembles)

## 7. Leaderboard

In [ ]:
leaderboard = pd.DataFrame(results).sort_values('macro_f1', ascending=False).reset_index(drop=True)
print(leaderboard.to_string(index=False))

plt.figure(figsize=(9, 5))
sns.barplot(data=leaderboard, x='macro_f1', y='model', color='seagreen')
plt.title('Macro-F1 by model (official test set)')
plt.xlabel('macro F1')
plt.tight_layout()
plt.show()

## 8. Best model detail — confusion matrix & per-class report

In [ ]:
available = [m for m in ['LightGBM', 'XGBoost', 'CatBoost'] if m in test_probs]
if available:
    soft_avg = np.mean([test_probs[m] for m in available], axis=0)
    y_pred_best = soft_avg.argmax(axis=1)
    inv_target_map = {v: k for k, v in target_map.items()}
    labels_order = sorted(inv_target_map)
    label_names = [inv_target_map[i] for i in labels_order]

    print(classification_report(y_test, y_pred_best, labels=labels_order, target_names=label_names))

    cm = confusion_matrix(y_test, y_pred_best, labels=labels_order)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion matrix — soft-voting ensemble')
    plt.tight_layout()
    plt.show()
else:
    print('No boosting models available for detailed report.')

## 9. Summary

- Baseline tier (Dummy, Logistic Regression, Decision Tree, Random Forest) trained on a 1.5M-row stratified subsample for tractable runtime.
- Boosting tier (LightGBM, XGBoost, CatBoost) trained on the full ~11.6M-row training set with early stopping against a 10% validation split.
- All models scored on the same untouched official `GUIDE_Test.csv`.
- Ensemble combines the three boosting models via soft voting and validation-F1-weighted voting.

Next step: hyperparameter tuning and/or a stacked meta-learner on out-of-fold base-model predictions.